<a href="https://colab.research.google.com/github/shrivash-raha/gemma-3-1b-lora-tuning/blob/main/gemma-3-1b-fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes torchao>=0.16.0

# Fix torchao version


In [48]:
!pip install -upgrade trl


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u


In [1]:
import trl
print(trl.__version__)

1.6.0


In [ ]:
!pip install -U torchao
print(torchao.__version__)

0.10.0


In [4]:
import torchao
print(torchao.__version__)


0.17.0


In [ ]:
!pip install --upgrade torchao>=0.16.0

In [ ]:
import peft
print(peft.__version__)

0.19.1


# Model download

In [1]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

from datasets import load_dataset

In [2]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

# Model Inspection

In [ ]:
total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Total Parameters: {total_params:,}")

Total Parameters: 496,195,456


In [ ]:
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable Parameters: {trainable_params:,}")

Trainable Parameters: 2,162,688


In [ ]:
for name, module in model.named_modules():
    if "proj" in name:
        print(name)

base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.base_layer
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_embedd

In [ ]:
for name, module in model.named_modules():
    if "proj" in name:
        print(f"Module: {name}")
        for param_name, param in module.named_parameters():
            print(f"  Parameter: {param_name}, Shape: {param.shape}")

Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj
  Parameter: base_layer.weight, Shape: torch.Size([896, 896])
  Parameter: base_layer.bias, Shape: torch.Size([896])
  Parameter: lora_A.default.weight, Shape: torch.Size([16, 896])
  Parameter: lora_B.default.weight, Shape: torch.Size([896, 16])
Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.base_layer
  Parameter: weight, Shape: torch.Size([896, 896])
  Parameter: bias, Shape: torch.Size([896])
Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout
Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default
Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A
  Parameter: default.weight, Shape: torch.Size([16, 896])
Module: 

# LoRA Config

In [3]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [4]:
model = get_peft_model(
    model,
    lora_config
)

# LoRA Inspection

In [ ]:
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([896, 16])
base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.Size([128, 16])
base_model.model

In [ ]:
for name, module in model.named_modules():
    if "q_proj" in name:
        print(name)
        print(type(module))
        break

base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj
<class 'peft.tuners.lora.layer.Linear'>


In [ ]:
layer = model.base_model.model.model.layers[0]

print(layer.self_attn.q_proj.lora_A["default"].weight.shape)
print(layer.self_attn.q_proj.lora_B["default"].weight.shape)

torch.Size([16, 896])
torch.Size([896, 16])


In [ ]:
A = layer.self_attn.q_proj.lora_A["default"].weight
B = layer.self_attn.q_proj.lora_B["default"].weight

print(A.shape)
print(B.shape)

delta_w = B @ A

print(delta_w.shape)

torch.Size([16, 896])
torch.Size([896, 16])
torch.Size([896, 896])


# Pre Training Inference

In [5]:
prompt = "Explain what LoRA is."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Explain what LoRA is. LoRA stands for Long Range Association and is a type of machine learning algorithm used in natural language processing (NLP) to identify long-range associations between words or phrases in text data. It was introduced by the University of Cambridge, UK, and has been shown to be effective at identifying relationships that are not immediately obvious from the context of the text.
LoRA works by analyzing the frequency distribution of word pairs within a document and then using this information to predict the likelihood of future word pairs occurring together.


# Dataset

In [6]:
dataset = load_dataset(
    "yahma/alpaca-cleaned"
)

dataset = dataset["train"].select(
    range(1000)
)

In [7]:
# def format_example(example):

#     text = f"""
# ### Instruction:
# {example['instruction']}

# ### Input:
# {example['input']}

# ### Response:
# {example['output']}
# """

#     return {"text": text}


def format_example(example):
    messages = [
        {
            "role": "user",
            "content": f"{example['instruction']}\n\n{example['input']}"
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    return {"messages": messages}

In [8]:
dataset = dataset.map(format_example)

# Dataset Exploration

In [12]:
dataset = load_dataset("yahma/alpaca-cleaned")

In [36]:
ds_w_ip = []

for kv in dataset['train']:
  if kv['input'] != '':
    ds_w_ip.append(kv)

In [35]:
for kv in ds_w_ip[:10]:
  print('\n--------------------------instruction--------------------------')
  print(kv['instruction'])
  print('\n--------------------------input--------------------------')
  print(kv['input'])
  print('\n--------------------------output--------------------------')
  print(kv['output'])
  print('\n\n========================================================================================================================================\n\n')





--------------------------instruction--------------------------
Explain why the following fraction is equivalent to 1/4

--------------------------input--------------------------
4/16

--------------------------output--------------------------
The fraction 4/16 is equivalent to 1/4 because both fractions represent the same value. A fraction can be simplified by dividing both the numerator and the denominator by a common factor. In this case, 4 is a common factor of both the numerator and the denominator of 4/16. When we divide both by 4, we get 4/4 = 1 and 16/4 = 4, so the simplified fraction is 1/4. Alternatively, we can think of this in terms of multiplication. For example, if we multiply the numerator and denominator of the fraction 1/4 by 4, we get (1x4)/(4x4), or 4/16. Since both fractions can be derived from the other through multiplication or division by the same number, they represent the same value and are equivalent.





--------------------------instruction----------------

# Training

In [9]:
from trl import SFTTrainer, SFTConfig

In [10]:
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_steps=100,
    learning_rate=2e-4,
    fp16=True,
    assistant_only_loss=True,
)

/tmp/ipykernel_37717/3155919270.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


In [11]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config
)

In [12]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.268631
20,1.214376
30,1.258422
40,1.316791
50,1.452363
60,1.251337
70,1.244051
80,1.180190
90,1.247380
100,1.223697


TrainOutput(global_step=125, training_loss=1.2654046707153321, metrics={'train_runtime': 186.514, 'train_samples_per_second': 5.362, 'train_steps_per_second': 0.67, 'total_flos': 540099841459200.0, 'train_loss': 1.2654046707153321, 'entropy': 1.1786759302020073, 'mean_token_accuracy': 0.6791499614715576, 'num_tokens': 181830.0, 'epoch': 1.0})

In [14]:
model.save_pretrained(
    "qwen_lora_adapter"
)

tokenizer.save_pretrained(
    "qwen_lora_adapter"
)

('qwen_lora_adapter/tokenizer_config.json',
 'qwen_lora_adapter/chat_template.jinja',
 'qwen_lora_adapter/tokenizer.json')

# Inference

In [15]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [20]:
model = PeftModel.from_pretrained(
    base_model,
    "qwen_lora_adapter"
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [17]:
prompt = "Explain what LoRA is."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100000
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Explain what LoRA is. LoRA stands for Low-Rank Matrix Approximation and it is a popular algorithm in the field of machine learning, particularly used for tasks such as image recognition and speech recognition.

LoRA is an iterative method that uses low-rank matrix approximations to solve large-scale optimization problems. It was developed by Google researchers and is designed to be efficient and effective at solving complex optimization problems with large datasets.

The core idea behind LoRA is to approximate a large matrix using smaller matrices called low-rank matrices, which have lower dimensions but are still close enough to the original matrix to perform well in many cases. The resulting approximation can then be used to find the optimal solution to the problem.

One key advantage of LoRA over other algorithms is its ability to handle high-dimensional data efficiently, making it suitable for tasks such as image and speech recognition where dealing with large datasets is critical.

'Qwen/Qwen2.5-0.5B-Instruct'